### Create a baseline to benchmark models on the Oct22 screening data

In [1]:
import pathlib

import numpy as np
import pandas as pd
import torch

In [2]:
data_dir = pathlib.Path("../output")
data_csv_fn = data_dir / "ordinal_Oct22_sequences_with_dca_score.csv"

In [40]:
protein_letters = 'ACDEFGHIKLMNPQRSTVWY'
aa_map = {a:i for i, a in enumerate(protein_letters)}
q = len(protein_letters)

prot_to_list = lambda x: [aa_map[xi] for xi in x]

def one_hot_encode_list(l):
  int_seqs = np.array([prot_to_list(x) for x in l], dtype=int)
  return np.eye(q)[int_seqs].reshape(int_seqs.shape[0], -1)

In [29]:
class Oct22DataSet:
    
    """ Wrapper for the sequences dataset
    
        1. Read in the sequences dataset 
        2. Lightly preprocess
        3. Read in train/test splits (and cv splits if exist)
    """
    
    NUM_CV_SPLITS = 5 # load upto these many cv splits
    
    def __init__(self, data_csv_fn):
        self.df = self._parse_df(data_csv_fn)
        self.L = len(self.df.sequence_aa_trim.iloc[0])
        self.train_indices = self._load_indices(data_csv_fn, "train")
        self.test_indices = self._load_indices(data_csv_fn, "test")
        
        # load in cv splits if they exist
        self.num_cv_splits = 0
        self.train_cv_splits = []
        self.val_cv_splits = []     
        self.load_cv_splits(data_csv_fn)
        
    def load_cv_splits(self, data_csv_fn):
        """ Load cv split indices. If no cv splits exist fail silently"""
        for i in range(self.NUM_CV_SPLITS):
            try:
                train_idx_cv = self._load_indices(data_csv_fn, f"train_cv{i+1}")
                val_idx_cv = self._load_indices(data_csv_fn, f"val_cv{i+1}")
            except FileNotFoundError:
                print(f"File not found train_cv{i+1} or val_cv{i+1}. Ignoring...")
                pass
            else:
                self.train_cv_splits.append(train_idx_cv)
                self.val_cv_splits.append(val_idx_cv)
        self.num_cv_splits = len(self.train_cv_splits)
        
    def get_train_dataset(self):
        return self.df.iloc[self.train_indices]
    
    def get_test_dataset(self):
        return self.df.iloc[self.test_indices]
    
    def cv_iterator(self):
        for i in self.num_cv_splits():
            yield (self.df.iloc[self.train_cv_splits[i]], #train cv split
                   self.df.iloc[self.val_cv_splits[i]]) # val cv split
    
    def __str__(self):
        return (f"aa_length :{self.L}\n" 
                f"num_seqs  :{len(self.df)}\n" 
                f"num_train :{len(self.train_indices)}\n"
                f"test_val  :{len(self.test_indices)}\n"
               )
        
    @staticmethod
    def _load_indices(data_csv_fn, idx_type):
        return np.loadtxt(data_csv_fn.with_suffix(f".{idx_type}.txt"), dtype=int)
        
    @staticmethod
    def _parse_df(data_csv_fn):
        df = pd.read_csv(data_csv_fn)
        df["library_num"] = df.parent.str.get(0).astype(int) - 1
        
        # map category to category code
        category_map = {'N':0, 'L':1, "P":2, "H":3}
        df["category_num"] = df["category"].map(category_map)
        
        # map activity to activity code (is_in_non_active_bin?)
        activity_map = {'N':0, 'L':1, "P":1, "H":1}
        df["activity_num"] = df["category"].map(activity_map)
        return df

In [32]:
ds = Oct22DataSet(data_csv_fn=data_csv_fn)
print(ds)

aa_length :272
num_seqs  :1326
num_train :1127
test_val  :199



In [5]:
ds.get_train_dataset()

,parent,category,response,dca_score,sequence_aa_trim,library_num,category_num,activity_num
492,2L,L,1,-452.495796,QHTYPAQQMRFGTAARAEHMTIAAAIHALDADVADAIVMDIVPDGE...,1,1,1
1316,3VRL,N,0,-441.910504,QHTYPAQLMRFGTAARAEHMTIAAAIHALDADGADAVVMDIVPDGE...,2,0,0
1172,3VRL,N,0,-438.657012,QHTYPAQLMRFGTAARAEHMTIAAAIHALDADEADAVVMDIVPDGE...,2,0,0
325,1VH,N,0,-446.571940,QHTYPAQLMRFGSAARAEHMTIAAAIHALDADEADAIVMDIVPDGE...,0,0,0
206,1VH,L,1,-452.035881,QHTYPAQLMRFGTAARAEHMTIAAAIHALDADEADAIVMDIVPDGE...,0,1,1
...,...,...,...,...,...,...,...,...
221,1VH,L,1,-454.164542,QHTYPAQLMRFGTAARAEHMTIAAAIHALDADEADAIVMDIVPDGE...,0,1,1
1253,3VRL,N,0,-441.792699,QHTYPAQLMRFGTAARAEHMTIAAAIHALDADEADAVVMDIVPDGE...,2,0,0
723,2L,L,1,-445.086206,QHTYPAQLMRFGTAARAEHMTIAAAIHALDADEADAIVMDIVPDGE...,1,1,1
909,3VRL,H,3,-446.547989,QHTYPAQLMRFGTAARAEHMTIAAAIHALDADEADAVVMDIVPDGE...,2,3,1


In [71]:
def create_model_inputs(df, add_dca=False, ):
  # add library num?
  design_matrix = one_hot_encode_list(df.sequence_aa_trim)
  if add_dca: # add dca to the last column
    design_matrix = np.hstack((design_matrix, df.dca_score.to_numpy()[:, np.newaxis]))
  
  return design_matrix

In [45]:
ret = create_model_inputs(ds.get_train_dataset())

In [46]:
df = ds.get_train_dataset()

In [63]:
ret.shape

(1127, 5440)

In [67]:
df.dca_score.to_numpy()[:, np.newaxis].shape

(1127, 1)

In [70]:
.shape

(1127, 5441)

In [62]:
np.vstack([ret, df.dca_score.to_numpy().transpose()]).shape

ValueError: all the input array dimensions for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 5440 and the array at index 1 has size 1127

In [52]:
df.dca_score.to_numpy().shape

(1127,)

In [53]:
ret.shape

(1127, 5440)

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])